# STC-HGAT PM2.5 Forecasting - Complete Training Pipeline

## Model Architecture Overview

**STC-HGAT**: Spatio-Temporal Contrastive Heterogeneous Graph Attention Network

### Key Components:
1. **HyperGAT** - Spatial heterogeneous hypergraph attention
   - Station proximity hyperedges
   - Geographic region nodes (5 regions)
   - Two-stage attention: nodes → hyperedges → nodes

2. **HGAT** - Temporal heterogeneous graph attention
   - Sequential day connections
   - Seasonal pattern edges
   - Temporal dependencies

3. **Fusion** - Spatio-temporal integration
   - Sum pooling: h = h_spatial + h_temporal
   - Position encoding (cyclical day-of-year)

4. **Loss Functions**:
   - Adaptive Weight Loss (upweight extreme PM2.5 events)
   - Contrastive Learning (InfoNCE for sparse haze events)
   - Combined: L = Lr + λ·Lc

### Fixed Issues:
- ✅ True graph attention (not plain self-attention)
- ✅ Fixed station-to-index mapping
- ✅ Date-based train/val/test split (no temporal leakage)
- ✅ Haversine distance for spatial graphs
- ✅ Proper gradient handling

## 1. Setup and Imports

In [16]:
# Auto-reload modules
%load_ext autoreload
%autoreload 2

import sys
import os
from pathlib import Path

# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

print(f"Project root: {project_root}")
print(f"Python path: {sys.path[0]}")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Project root: /home/supawich/Desktop/bkk-pm25-data-ingestion
Python path: /home/supawich/Desktop/bkk-pm25-data-ingestion


In [17]:
# Core imports
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import yaml
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns

# Project imports
from src.models.stc_hgat_model import STCHGAT
from src.data.dataset import split_by_date, PM25SequenceDataset
from src.utils.graph_builder import (
    build_spatial_hypergraph,
    build_temporal_graph,
    compute_region_embeddings,
    haversine_km
)
from src.utils.evaluator import calculate_mae, calculate_rmse, calculate_r2
from src.utils.mlflow_config import setup_mlflow

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ All imports successful!")

✅ All imports successful!


## 2. Check Project Structure and Readiness

In [18]:
# Check project structure
print("📁 Project Structure Check:")
print("="*50)

required_files = [
    'src/models/stc_hgat_model.py',
    'src/data/dataset.py',
    'src/utils/graph_builder.py',
    'src/utils/evaluator.py',
    'params.yaml',
    'config.yaml',
    'data/stations/bangkok_stations.parquet'
]

all_ready = True
for file in required_files:
    file_path = project_root / file
    exists = file_path.exists()
    status = "✅" if exists else "❌"
    print(f"{status} {file}")
    if not exists:
        all_ready = False

print("\n" + "="*50)
if all_ready:
    print("✅ All required files present!")
else:
    print("❌ Some files are missing!")
    raise FileNotFoundError("Required files missing")

📁 Project Structure Check:
✅ src/models/stc_hgat_model.py
✅ src/data/dataset.py
✅ src/utils/graph_builder.py
✅ src/utils/evaluator.py
✅ params.yaml
✅ config.yaml
✅ data/stations/bangkok_stations.parquet

✅ All required files present!


In [19]:
# Check GPU/Device availability
print("\n🖥️  Device Check:")
print("="*50)

if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f"✅ CUDA available: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = torch.device('mps')
    print("✅ Apple MPS available")
else:
    device = torch.device('cpu')
    print("⚠️  Using CPU (training will be slower)")

print(f"\nSelected device: {device}")
print("="*50)


🖥️  Device Check:
✅ CUDA available: NVIDIA GeForce RTX 3080 Ti
   Memory: 12.49 GB

Selected device: cuda


## 3. Load Configuration

In [20]:
# Load parameters
with open(project_root / 'params.yaml') as f:
    params = yaml.safe_load(f)

print("⚙️  Configuration:")
print("="*50)
print(f"Model hidden_dim: {params['model']['hidden_dim']}")
print(f"Batch size: {params['training']['batch_size']}")
print(f"Learning rate: {params['training']['learning_rate']}")
print(f"Epochs: {params['training']['epochs']}")
print(f"Sequence length: {params['data']['sequence_length']}")
print(f"Forecast horizons: {params['data']['forecast_horizons']}")
print("="*50)

⚙️  Configuration:
Model hidden_dim: 128
Batch size: 32
Learning rate: 0.001
Epochs: 100
Sequence length: 30
Forecast horizons: [1, 3, 7]


## 4. Load and Explore Station Data

In [ ]:
# Load station metadata
stations_df = pd.read_parquet(project_root / 'data/stations/bangkok_stations.parquet')

print("🗺️  Station Data:")
print("="*50)
print(f"Total stations: {len(stations_df)}")
print(f"\nColumns: {list(stations_df.columns)}")
print(f"\nFirst 5 stations:")
print(stations_df.head())

# Visualize station locations
plt.figure(figsize=(10, 8))
plt.scatter(stations_df['lon'], stations_df['lat'], alpha=0.6, s=100)
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.title(f'Bangkok PM2.5 Monitoring Stations (n={len(stations_df)})')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("="*50)

Note: you may need to restart the kernel to use updated packages.


In [22]:
import pandas as pd
import matplotlib.pyplot as plt

# =========================
# 1. LOAD DATA
# =========================
from pathlib import Path

project_root = Path(".")  # แก้ path ตามโปรเจกต์คุณ
stations_df = pd.read_parquet(project_root / 'data/stations/bangkok_stations.parquet')

print("🗺️  Station Data:")
print("="*50)
print(f"Total stations: {len(stations_df)}")
print(f"Columns: {list(stations_df.columns)}")
print(stations_df.head())


# =========================
# 2. FILTER กรุงเทพ (optional แต่แนะนำ)
# =========================
bbox = {
    "lat_min": 13.4,
    "lat_max": 14.1,
    "lon_min": 100.3,
    "lon_max": 100.9
}

stations_df = stations_df[
    (stations_df['lat'] >= bbox["lat_min"]) &
    (stations_df['lat'] <= bbox["lat_max"]) &
    (stations_df['lon'] >= bbox["lon_min"]) &
    (stations_df['lon'] <= bbox["lon_max"])
]

print("\nAfter filtering Bangkok area:")
print(f"Total stations: {len(stations_df)}")


# =========================
# 3. INTERACTIVE MAP (Folium)
# =========================
import folium

center_lat = stations_df['lat'].mean()
center_lon = stations_df['lon'].mean()

m = folium.Map(location=[center_lat, center_lon], zoom_start=11)

for _, row in stations_df.iterrows():
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=6,
        popup=f"Station: {row.get('station_id', 'unknown')}",
        color='blue',
        fill=True,
        fill_opacity=0.7
    ).add_to(m)

# save map
m.save("bangkok_pm25_map.html")

print("\n✅ Saved interactive map: bangkok_pm25_map.html")


# =========================
# 4. STATIC MAP (Matplotlib + Contextily)
# =========================
import geopandas as gpd
import contextily as ctx

# convert to GeoDataFrame
gdf = gpd.GeoDataFrame(
    stations_df,
    geometry=gpd.points_from_xy(stations_df.lon, stations_df.lat),
    crs="EPSG:4326"
)

# convert CRS for basemap
gdf = gdf.to_crs(epsg=3857)

fig, ax = plt.subplots(figsize=(10, 8))

gdf.plot(ax=ax, alpha=0.7, markersize=50)

# add basemap
ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron)

ax.set_title(f'Bangkok PM2.5 Monitoring Stations (n={len(gdf)})')
ax.set_axis_off()

plt.tight_layout()
plt.show()


# =========================
# 5. (OPTIONAL) HEATMAP
# =========================
from folium.plugins import HeatMap

heat_data = [[row['lat'], row['lon']] for _, row in stations_df.iterrows()]

heat_map = folium.Map(location=[center_lat, center_lon], zoom_start=11)
HeatMap(heat_data).add_to(heat_map)

heat_map.save("bangkok_pm25_heatmap.html")

print("🔥 Saved heatmap: bangkok_pm25_heatmap.html")

FileNotFoundError: [Errno 2] No such file or directory: 'data/stations/bangkok_stations.parquet'

## 5. Build Spatial and Temporal Graphs

In [23]:
print("🕸️  Building Graphs:")
print("="*50)

# Build spatial hypergraph
print("Building spatial hypergraph...")
spatial_graph = build_spatial_hypergraph(
    stations_df,
    threshold_km=params['graph']['spatial_threshold_km']
)

if 'hyperedge_index' in spatial_graph:
    print(f"✅ Spatial hypergraph: {spatial_graph['hyperedge_index'].shape[1]} connections")
elif 'edge_index' in spatial_graph:
    print(f"✅ Spatial graph: {spatial_graph['edge_index'].shape[1]} edges")

# Build temporal graph
print("\nBuilding temporal graph...")
temporal_graph = build_temporal_graph(
    num_days=365,
    seasonal_pattern=params['graph']['seasonal_pattern']
)
print(f"✅ Temporal graph: {temporal_graph['edge_index'].shape[1]} edges")

# Compute region embeddings
print("\nComputing region embeddings...")
region_embeddings = compute_region_embeddings(
    stations_df,
    num_regions=params['graph']['num_regions']
)
print(f"✅ Region embeddings: {region_embeddings.shape}")

print("="*50)

🕸️  Building Graphs:
Building spatial hypergraph...
✅ Spatial graph: 6130 edges

Building temporal graph...
✅ Temporal graph: 19032 edges

Computing region embeddings...
✅ Region embeddings: torch.Size([5, 2])


## 6. Create Mock Data for Training Demo

**Note**: In production, replace this with actual PM2.5 data loading from bronze/silver layers

In [24]:
print("📊 Creating Mock Training Data:")
print("="*50)
print("⚠️  Using synthetic data for demonstration")
print("   In production: Load from data/silver/ with actual PM2.5 measurements")
print()

# Create synthetic data
num_stations = len(stations_df)
num_timesteps = 1000  # ~3 years of daily data
num_features = len(params['data']['features'])

# Generate realistic-looking time series
np.random.seed(42)
torch.manual_seed(42)

# Base pattern + noise + seasonal component
t = np.arange(num_timesteps)
seasonal = 20 * np.sin(2 * np.pi * t / 365)  # Yearly pattern
trend = 0.01 * t  # Slight trend
noise = np.random.randn(num_timesteps, num_stations, num_features) * 10

data = torch.zeros(num_timesteps, num_stations, num_features)
for i in range(num_features):
    base = 50 + seasonal[:, None] + trend[:, None]  # Base PM2.5 ~50 µg/m³
    data[:, :, i] = torch.tensor(base + noise[:, :, i], dtype=torch.float32)

# Ensure positive values
data = torch.clamp(data, min=0, max=500)

print(f"Data shape: {data.shape}")
print(f"  Timesteps: {num_timesteps}")
print(f"  Stations: {num_stations}")
print(f"  Features: {num_features}")
print(f"\nData statistics:")
print(f"  Mean: {data.mean():.2f}")
print(f"  Std: {data.std():.2f}")
print(f"  Min: {data.min():.2f}")
print(f"  Max: {data.max():.2f}")
print("="*50)

📊 Creating Mock Training Data:
⚠️  Using synthetic data for demonstration
   In production: Load from data/silver/ with actual PM2.5 measurements

Data shape: torch.Size([1000, 79, 9])
  Timesteps: 1000
  Stations: 79
  Features: 9

Data statistics:
  Mean: 56.24
  Std: 17.15
  Min: 0.00
  Max: 124.98


## 7. Create Train/Val/Test Splits

In [25]:
print("✂️  Creating Data Splits:")
print("="*50)

# Split data chronologically (no temporal leakage)
train_ratio = params['data']['train_ratio']
val_ratio = params['data']['val_ratio']

n = len(data)
train_end = int(n * train_ratio)
val_end = int(n * (train_ratio + val_ratio))

train_data = data[:train_end]
val_data = data[train_end:val_end]
test_data = data[val_end:]

print(f"Train: {len(train_data)} timesteps ({train_ratio*100:.0f}%)")
print(f"Val:   {len(val_data)} timesteps ({val_ratio*100:.0f}%)")
print(f"Test:  {len(test_data)} timesteps ({(1-train_ratio-val_ratio)*100:.0f}%)")

# Normalize using training statistics only
train_mean = train_data.mean(dim=(0, 1), keepdim=True)
train_std = train_data.std(dim=(0, 1), keepdim=True) + 1e-8

train_data_norm = (train_data - train_mean) / train_std
val_data_norm = (val_data - train_mean) / train_std
test_data_norm = (test_data - train_mean) / train_std

print(f"\nNormalization stats (from training data):")
print(f"  Mean: {train_mean.squeeze().mean():.2f}")
print(f"  Std: {train_std.squeeze().mean():.2f}")
print("="*50)

✂️  Creating Data Splits:
Train: 700 timesteps (70%)
Val:   150 timesteps (15%)
Test:  150 timesteps (15%)

Normalization stats (from training data):
  Mean: 53.71
  Std: 17.00


## 8. Create PyTorch Datasets and DataLoaders

In [26]:
print("🔄 Creating DataLoaders:")
print("="*50)

seq_len = params['data']['sequence_length']
horizons = params['data']['forecast_horizons']
batch_size = params['training']['batch_size']

# Create datasets
train_dataset = PM25SequenceDataset(
    train_data_norm,
    sequence_length=seq_len,
    forecast_horizons=horizons
)

val_dataset = PM25SequenceDataset(
    val_data_norm,
    sequence_length=seq_len,
    forecast_horizons=horizons
)

test_dataset = PM25SequenceDataset(
    test_data_norm,
    sequence_length=seq_len,
    forecast_horizons=horizons
)

# Create dataloaders
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0
)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")
print(f"\nBatch size: {batch_size}")
print(f"Sequence length: {seq_len}")
print(f"Forecast horizons: {horizons}")
print("="*50)

🔄 Creating DataLoaders:
Train batches: 21
Val batches: 4
Test batches: 4

Batch size: 32
Sequence length: 30
Forecast horizons: [1, 3, 7]


## 9. Initialize STC-HGAT Model

In [27]:
print("🤖 Initializing STC-HGAT Model:")
print("="*50)

model = STCHGAT(
    num_features=num_features,
    hidden_dim=params['model']['hidden_dim'],
    num_stations=num_stations,
    num_regions=params['graph']['num_regions'],
    num_hypergat_layers=params['model']['num_hypergat_layers'],
    num_hgat_layers=params['model']['num_hgat_layers'],
    num_heads=params['model']['num_heads'],
    dropout=params['model']['dropout'],
    forecast_horizons=horizons
).to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Model: STC-HGAT")
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"\nModel configuration:")
print(f"  Hidden dim: {params['model']['hidden_dim']}")
print(f"  HyperGAT layers: {params['model']['num_hypergat_layers']}")
print(f"  HGAT layers: {params['model']['num_hgat_layers']}")
print(f"  Attention heads: {params['model']['num_heads']}")
print(f"  Dropout: {params['model']['dropout']}")
print(f"\nDevice: {device}")
print("="*50)

🤖 Initializing STC-HGAT Model:
Model: STC-HGAT
Total parameters: 243,457
Trainable parameters: 243,457

Model configuration:
  Hidden dim: 128
  HyperGAT layers: 2
  HGAT layers: 2
  Attention heads: 4
  Dropout: 0.2

Device: cuda


## 10. Setup Training Components

In [28]:
print("⚙️  Setup Training:")
print("="*50)

# Optimizer
optimizer = optim.AdamW(
    model.parameters(),
    lr=params['training']['learning_rate'],
    weight_decay=params['training']['weight_decay']
)

# Scheduler
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=params['training']['scheduler_factor'],
    patience=params['training']['scheduler_patience'],
    verbose=True
)

# Loss function
criterion = nn.MSELoss()

# Move graphs to device
spatial_graph_device = {k: v.to(device) for k, v in spatial_graph.items()}
temporal_graph_device = {k: v.to(device) for k, v in temporal_graph.items()}

print(f"Optimizer: AdamW")
print(f"  Learning rate: {params['training']['learning_rate']}")
print(f"  Weight decay: {params['training']['weight_decay']}")
print(f"\nScheduler: ReduceLROnPlateau")
print(f"  Patience: {params['training']['scheduler_patience']}")
print(f"  Factor: {params['training']['scheduler_factor']}")
print(f"\nLoss: MSE")
print(f"Gradient clipping: {params['training']['gradient_clip_value']}")
print("="*50)

⚙️  Setup Training:


/home/supawich/.venvs/stc_hgat_pm25_forecasting/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


AttributeError: 'list' object has no attribute 'to'

## 11. Training Loop

In [ ]:
print("🚀 Starting Training:")
print("="*50)

num_epochs = params['training']['epochs']
early_stopping_patience = params['training']['early_stopping_patience']
gradient_clip = params['training']['gradient_clip_value']

# Training history
history = {
    'train_loss': [],
    'val_loss': [],
    'learning_rate': []
}

best_val_loss = float('inf')
patience_counter = 0
best_model_state = None

for epoch in range(num_epochs):
    # Training phase
    model.train()
    train_loss = 0.0
    
    for batch_idx, (X, y) in enumerate(train_loader):
        X, y = X.to(device), y.to(device)
        
        optimizer.zero_grad()
        
        # Forward pass
        output = model(X, spatial_graph_device, temporal_graph_device)
        loss = criterion(output, y)
        
        # Backward pass
        loss.backward()
        
        # Gradient clipping
        if gradient_clip:
            torch.nn.utils.clip_grad_norm_(model.parameters(), gradient_clip)
        
        optimizer.step()
        
        train_loss += loss.item()
    
    train_loss /= len(train_loader)
    
    # Validation phase
    model.eval()
    val_loss = 0.0
    
    with torch.no_grad():
        for X, y in val_loader:
            X, y = X.to(device), y.to(device)
            output = model(X, spatial_graph_device, temporal_graph_device)
            loss = criterion(output, y)
            val_loss += loss.item()
    
    val_loss /= len(val_loader)
    
    # Update learning rate
    scheduler.step(val_loss)
    current_lr = optimizer.param_groups[0]['lr']
    
    # Save history
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['learning_rate'].append(current_lr)
    
    # Early stopping check
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        best_model_state = model.state_dict().copy()
        print(f"Epoch {epoch+1}/{num_epochs} - Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f} ⭐ (Best)")
    else:
        patience_counter += 1
        print(f"Epoch {epoch+1}/{num_epochs} - Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")
    
    if patience_counter >= early_stopping_patience:
        print(f"\n⏹️  Early stopping triggered after {epoch+1} epochs")
        break

# Load best model
if best_model_state:
    model.load_state_dict(best_model_state)
    print(f"\n✅ Loaded best model (val_loss: {best_val_loss:.4f})")

print("="*50)
print("Training completed!")

## 12. Visualize Training Progress

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Loss curves
axes[0].plot(history['train_loss'], label='Train Loss', linewidth=2)
axes[0].plot(history['val_loss'], label='Val Loss', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss (MSE)')
axes[0].set_title('Training and Validation Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Learning rate
axes[1].plot(history['learning_rate'], linewidth=2, color='green')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Learning Rate')
axes[1].set_title('Learning Rate Schedule')
axes[1].set_yscale('log')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n📊 Training Summary:")
print(f"  Best val loss: {best_val_loss:.4f}")
print(f"  Final train loss: {history['train_loss'][-1]:.4f}")
print(f"  Total epochs: {len(history['train_loss'])}")

## 13. Evaluate on Test Set

In [ ]:
print("📈 Test Set Evaluation:")
print("="*50)

model.eval()
test_predictions = []
test_targets = []

with torch.no_grad():
    for X, y in test_loader:
        X, y = X.to(device), y.to(device)
        output = model(X, spatial_graph_device, temporal_graph_device)
        test_predictions.append(output.cpu())
        test_targets.append(y.cpu())

test_predictions = torch.cat(test_predictions, dim=0)
test_targets = torch.cat(test_targets, dim=0)

# Calculate metrics for each horizon
print("\nMetrics by Forecast Horizon:")
for i, horizon in enumerate(horizons):
    pred_h = test_predictions[:, :, i]
    target_h = test_targets[:, :, i]
    
    mae = calculate_mae(target_h, pred_h)
    rmse = calculate_rmse(target_h, pred_h)
    r2 = calculate_r2(target_h, pred_h)
    
    print(f"\nHorizon +{horizon}d:")
    print(f"  MAE:  {mae:.4f}")
    print(f"  RMSE: {rmse:.4f}")
    print(f"  R²:   {r2:.4f}")

print("="*50)

## 14. Visualize Predictions

In [ ]:
# Plot predictions vs actual for first station and first horizon
station_idx = 0
horizon_idx = 0

pred = test_predictions[:, station_idx, horizon_idx].numpy()
actual = test_targets[:, station_idx, horizon_idx].numpy()

fig, axes = plt.subplots(2, 1, figsize=(15, 8))

# Time series
axes[0].plot(actual[:200], label='Actual', alpha=0.7, linewidth=2)
axes[0].plot(pred[:200], label='Predicted', alpha=0.7, linewidth=2)
axes[0].set_xlabel('Time Step')
axes[0].set_ylabel('Normalized PM2.5')
axes[0].set_title(f'Predictions vs Actual (Station {station_idx}, Horizon +{horizons[horizon_idx]}d)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Scatter plot
axes[1].scatter(actual, pred, alpha=0.5, s=20)
axes[1].plot([actual.min(), actual.max()], [actual.min(), actual.max()], 
             'r--', linewidth=2, label='Perfect prediction')
axes[1].set_xlabel('Actual')
axes[1].set_ylabel('Predicted')
axes[1].set_title('Prediction Scatter Plot')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 15. Save Model

In [ ]:
# Create models directory
models_dir = project_root / 'models'
models_dir.mkdir(exist_ok=True)

# Save model
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
model_path = models_dir / f'stc_hgat_{timestamp}.pt'

torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'history': history,
    'best_val_loss': best_val_loss,
    'params': params,
    'train_mean': train_mean,
    'train_std': train_std
}, model_path)

print(f"💾 Model saved to: {model_path}")
print(f"   Size: {model_path.stat().st_size / 1e6:.2f} MB")

## 16. Summary

In [ ]:
print("\n" + "="*60)
print("🎉 TRAINING COMPLETE - STC-HGAT PM2.5 Forecasting")
print("="*60)

print("\n📊 Model Architecture:")
print(f"  - HyperGAT layers: {params['model']['num_hypergat_layers']}")
print(f"  - HGAT layers: {params['model']['num_hgat_layers']}")
print(f"  - Hidden dimension: {params['model']['hidden_dim']}")
print(f"  - Total parameters: {total_params:,}")

print("\n📈 Training Results:")
print(f"  - Epochs trained: {len(history['train_loss'])}")
print(f"  - Best val loss: {best_val_loss:.4f}")
print(f"  - Final train loss: {history['train_loss'][-1]:.4f}")

print("\n🎯 Test Performance:")
for i, horizon in enumerate(horizons):
    pred_h = test_predictions[:, :, i]
    target_h = test_targets[:, :, i]
    mae = calculate_mae(target_h, pred_h)
    print(f"  - Horizon +{horizon}d: MAE = {mae:.4f}")

print("\n💾 Saved:")
print(f"  - Model: {model_path.name}")

print("\n✅ Next Steps:")
print("  1. Replace mock data with actual PM2.5 measurements")
print("  2. Run hyperparameter tuning: python src/tune.py")
print("  3. Track experiments with MLflow: mlflow ui")
print("  4. Run DVC pipeline: dvc repro")
print("  5. Deploy model for production forecasting")

print("\n" + "="*60)